# From Space to Action - Agricultural Drought Early Warning
## Notebook 02: Preprocessing
**Goal:** Clean, mask, composite, and harmonize datasets.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/FromSpaceToAction'
DATA_DIR = f'{PROJECT_DIR}/data'
SRC_DIR = f'{PROJECT_DIR}/src'
MODELS_DIR = f'{PROJECT_DIR}/models'
OUTPUTS_DIR = f'{PROJECT_DIR}/outputs'
CONFIG_PATH = f'{PROJECT_DIR}/config/config.yaml'

for d in [DATA_DIR, f'{DATA_DIR}/raw', f'{DATA_DIR}/processed', f'{DATA_DIR}/features', f'{DATA_DIR}/targets', MODELS_DIR, OUTPUTS_DIR, f'{OUTPUTS_DIR}/maps', f'{OUTPUTS_DIR}/reports']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, SRC_DIR)

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print("Loading libraries...")

### Load raw data (either from GEE exports or sample data)

In [ ]:
raw_data_path = f'{DATA_DIR}/raw/sample_data.parquet'
if os.path.exists(raw_data_path):
    df = pd.read_parquet(raw_data_path)
    print(f"Loaded raw data with shape: {df.shape}")
else:
    print("Raw data not found! Please run Notebook 01 first.")

### Cloud masking for optical imagery & Dekadal compositing

In [ ]:
# In a real scenario, cloud masking would occur in GEE or via QA bands.
# Here we simulate QA cleaning by setting random low NDVI to NaN
np.random.seed(42)
df.loc[df.sample(frac=0.05).index, 'ndvi'] = np.nan

# Group by 10-day intervals (dekads)
df['dekad'] = df['date'].dt.floor('10D')
df_dekadal = df.groupby(['dekad', 'lat', 'lon']).mean().reset_index()
print("Dekadal compositing complete.")

### Rainfall accumulation windows (7, 14, 30, 60 days)

In [ ]:
df_dekadal = df_dekadal.sort_values(by=['lat', 'lon', 'dekad'])

for window in [1, 2, 3, 6]: # in dekads, corresponding roughly to 10, 20, 30, 60 days
    col_name = f'rainfall_{window*10}d_acc'
    df_dekadal[col_name] = df_dekadal.groupby(['lat', 'lon'])['rainfall'].rolling(window=window, min_periods=1).sum().reset_index(0, drop=True)
    
print("Rainfall accumulations computed.")

### Rainfall anomaly computation (z-scores vs climatology)

In [ ]:
# Simulating climatology by dekad of year
df_dekadal['doy'] = df_dekadal['dekad'].dt.dayofyear
clim = df_dekadal.groupby(['lat', 'lon', 'doy'])['rainfall_30d_acc'].agg(['mean', 'std']).reset_index()
df_dekadal = pd.merge(df_dekadal, clim, on=['lat', 'lon', 'doy'], how='left')
df_dekadal['rainfall_30d_anomaly'] = (df_dekadal['rainfall_30d_acc'] - df_dekadal['mean']) / (df_dekadal['std'] + 1e-6)
df_dekadal.drop(columns=['mean', 'std', 'doy'], inplace=True)
print("Rainfall anomalies computed.")

### Soil moisture preprocessing & ERA5 temperature anomaly

In [ ]:
df_dekadal['smap_anomaly'] = (df_dekadal['soil_moisture'] - df_dekadal['soil_moisture'].mean()) / df_dekadal['soil_moisture'].std()
df_dekadal['temp_anomaly'] = (df_dekadal['temperature'] - df_dekadal['temperature'].mean()) / df_dekadal['temperature'].std()
print("Soil moisture and temperature anomalies computed.")

### Spatial harmonization & Quality filtering

In [ ]:
# Filter cells with < 50% valid observations for NDVI
valid_counts = df_dekadal.groupby(['lat', 'lon'])['ndvi'].count()
total_counts = df_dekadal.groupby(['lat', 'lon']).size()
valid_ratio = valid_counts / total_counts
valid_cells = valid_ratio[valid_ratio >= 0.5].index

df_harmonized = df_dekadal.set_index(['lat', 'lon']).loc[valid_cells].reset_index()
print(f"Retained {len(valid_cells)} valid spatial cells.")

### Missingness report with visualization

In [ ]:
missing = df_harmonized.isnull().mean() * 100
print("Missing Data Percentage:\n", missing[missing > 0])

plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='salmon')
plt.title('Percentage of Missing Values per Feature')
plt.ylabel('% Missing')
plt.show()

### Save cleaned data to data/processed/

In [ ]:
processed_path = f'{DATA_DIR}/processed/processed_data.parquet'
df_harmonized.to_parquet(processed_path)
print(f"Cleaned data saved to {processed_path}")